# BPv5 Stage A 监控（E1 + HOM）

只读 `loss.log`、`heldout_monitor.jsonl`、`g9/a_*.json`，不加载模型。

- **E1 / T1**（对齐 `damntwin/docs/测评报告模板.md` §4.1）：左训练 loss（EMA 200），右留出 matched 动作 MSE
- **C2 / C3**（对齐 `stage_b_compare_b3_b5.ipynb`）：matched 曲线；blank / alt / wrong 比值 + 95% CI 阴影

离线 HOM：`bash tools/bpv5/run_stage_a_hom_offline.sh <run_dir> 32`。Stage B 可继续沿用本 notebook 的 C2/C3 绘图函数。

In [ ]:
from pathlib import Path
import csv
import io
import json
from datetime import datetime, timezone

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string(index=False) if isinstance(obj, pd.DataFrame) else obj)

REPO = Path('/vla/workspace/my_tbot')
STAGE_A = REPO / 'outputs/BPv5/stage_a'

RUNS = [
    {'name': 'BPv5-local4', 'run_root': STAGE_A / 'main' / '2026-11-13_14-46', 'start_step': 0},
    # {'name': 'BPv5-dist8', 'run_root': STAGE_A / 'main_dist8' / '2026-11-13_22-56', 'start_step': 0},
]

SMOOTH_STEPS = 1000
EMA_SPAN = 200
EXPORT = True
EXPORT_DIR = STAGE_A / 'reports' / 'stage_a_e1_hom'
FORMATS = ('png', 'pdf', 'svg')
DPI = 300
FIGSIZE_ROW = (14, 4.3)
FIGSIZE_RATIO = (5.5, 4.6)

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.22,
    'legend.fontsize': 8, 'figure.dpi': 110, 'figure.max_open_warning': 0,
    'savefig.facecolor': 'white', 'pdf.fonttype': 42, 'svg.fonttype': 'none',
})

COLORS = {'matched': '#0072B2', 'blank': '#D55E00', 'alt': '#009E73', 'wrong': '#CC79A7'}
RUN_COLORS = {'BPv5-local4': '#D55E00', 'BPv5-dist8': '#009E73', 'smoke': '#0072B2'}
MARKERS = ('o', 's', '^', 'D')
LINESTYLES = ('-', '--', ':', '-.')
MSE_YLABEL = 'Action MSE (normalized delta)'
RATIO_YLABEL = 'Error ratio (unitless)'

In [ ]:
def resolve_run(root):
    root = Path(root).expanduser().resolve()
    if (root / 'loss.log').is_file():
        return root
    cands = sorted(p.parent for p in root.glob('*/loss.log'))
    if len(cands) > 1:
        raise ValueError(f'{root} 下有多个 run，请指定时间戳目录：{cands}')
    return cands[0].resolve() if cands else None


def load_loss(path):
    text = path.read_text(encoding='utf-8')
    text = text[: text.rfind('\n') + 1]
    rows = []
    for r in csv.DictReader(io.StringIO(text)):
        try:
            step, loss = int(r['step']), float(r['loss'])
        except (KeyError, ValueError, TypeError):
            continue
        if step < 1 or not np.isfinite(loss):
            continue
        rows.append({'step': step, 'loss': loss})
    return pd.DataFrame(rows).drop_duplicates('step', keep='last').sort_values('step').reset_index(drop=True)


def _ratio_entry(entry):
    if not isinstance(entry, dict):
        return np.nan, np.nan, np.nan
    ci = entry.get('ci95') or [np.nan, np.nan]
    return (
        entry.get('ratio', np.nan),
        ci[0] if len(ci) > 0 else np.nan,
        ci[1] if len(ci) > 1 else np.nan,
    )


def load_hom(run):
    rows = []
    path = run / 'heldout_monitor.jsonl'
    if path.is_file():
        for line in path.read_text().splitlines():
            if not line.strip():
                continue
            d = json.loads(line)
            if d.get('mse_matched') is None:
                continue
            row = {
                'step': int(d['step']),
                'queries': int(d.get('episodes') or 0),
                'matched': float(d['mse_matched']),
                'blank': float(d['mse_blank']) if d.get('mse_blank') is not None else np.nan,
                'alt': float(d['mse_alternate']) if d.get('mse_alternate') is not None else np.nan,
                'wrong': float(d['mse_wrong']) if d.get('mse_wrong') is not None else np.nan,
                'segments': d.get('segments'),
            }
            for cond in ('blank', 'alt', 'wrong'):
                ratio, lo, hi = _ratio_entry(d.get(f'{cond}_over_match') or {})
                row[f'{cond}_ratio'], row[f'{cond}_ci_low'], row[f'{cond}_ci_high'] = ratio, lo, hi
            rows.append(row)
    g9 = run / 'g9'
    if g9.is_dir():
        for p in sorted(g9.glob('a_*.json')):
            d = json.loads(p.read_text())
            step = int(d.get('step') or p.stem.split('_')[-1])
            if any(r['step'] == step for r in rows):
                continue
            me, ratios = d.get('mean_error') or {}, d.get('ratios') or {}
            row = {
                'step': step,
                'queries': int(d.get('queries') or d.get('episodes') or 0),
                'matched': me.get('matched', d.get('mse_matched')),
                'blank': me.get('blank', d.get('mse_blank')),
                'alt': me.get('alt', d.get('mse_alternate')),
                'wrong': me.get('wrong', d.get('mse_wrong')),
                'segments': d.get('segments'),
            }
            for cond in ('blank', 'alt', 'wrong'):
                ratio, lo, hi = _ratio_entry(ratios.get(cond) or d.get(f'{cond}_over_match') or {})
                row[f'{cond}_ratio'], row[f'{cond}_ci_low'], row[f'{cond}_ci_high'] = ratio, lo, hi
            rows.append(row)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).drop_duplicates('step', keep='last').sort_values('step').reset_index(drop=True)


def infer_freq(df):
    d = df['step'].diff().dropna()
    return 100 if d.empty else int(d.mode().iloc[0])


def train_smooth(df, metric, freq, width=SMOOTH_STEPS):
    k = width // freq
    series = df.set_index('step')[metric]
    steps = pd.Series(series.index, index=series.index)
    complete = steps - steps.shift(k - 1) == width - freq
    return series.rolling(k, min_periods=k).mean().where(complete).to_numpy()


def step_k(x, _=None):
    return f'{x / 1000:.0f}k' if abs(x) >= 1000 else f'{x:.0f}'


CTX = {}
for spec in RUNS:
    run = resolve_run(spec['run_root'])
    if run is None:
        print(f"{spec['name']}: 尚无 run（{spec['run_root']}）")
        continue
    train = load_loss(run / 'loss.log') if (run / 'loss.log').is_file() else pd.DataFrame()
    hom = load_hom(run)
    freq = infer_freq(train) if not train.empty else 100
    CTX[spec['name']] = {
        **spec, 'run': run, 'train': train, 'hom': hom, 'freq': freq,
        'start': int(spec.get('start_step', 0)),
    }
    print(
        f"{spec['name']} ({run.name}): loss→{0 if train.empty else int(train.step.iloc[-1]):,}; "
        f"HOM {len(hom)} ckpts" + (f" steps={hom.step.tolist()}" if len(hom) else '')
    )

FIGURES = {}
NOTES = []


def register(name, fig, note=''):
    FIGURES[name] = fig
    NOTES.append({'figure': name, 'caption': note})
    plt.show()


def finish_axis(ax, title, ylabel, xlabel):
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)
    ax.legend(frameon=False, loc='best')


def ratio_refs(ax, condition):
    ax.axhline(1.0, color='#666', ls='--', lw=1.0, label='no error change (1.0)')
    if condition == 'blank':
        ax.axhline(1.1, color='#999', ls=':', lw=1.0, label='G9 ratio threshold')

In [ ]:
# ===== E1 / T1: loss + matched MSE (测评报告模板 §4.1) =====
if CTX:
    fig, axs = plt.subplots(1, 2, figsize=FIGSIZE_ROW)
    for name, ctx in CTX.items():
        color = RUN_COLORS.get(name, '#333')
        tr, hom = ctx['train'], ctx['hom']
        if not tr.empty:
            ema = tr['loss'].ewm(span=EMA_SPAN).mean()
            axs[0].plot(tr['step'] / 1000, ema, color=color, lw=1.2, label=f'{name} EMA {EMA_SPAN}')
            # trailing-mean overlay (stage_b_compare style)
            sm = train_smooth(tr, 'loss', ctx['freq'])
            axs[0].plot(tr['step'] / 1000, sm, color=color, lw=1.0, alpha=0.55,
                        label=f'{name} trailing {SMOOTH_STEPS:,}')
        if not hom.empty:
            axs[1].plot(hom['step'] / 1000, hom['matched'], color=color, marker='o', ms=5, lw=1.6,
                        label=f'{name} held-out matched')
    axs[0].set(yscale='log', title='Training loss',
               xlabel='Step (k)', ylabel='Flow loss (unitless)')
    axs[0].legend(frameon=False, loc='upper right')
    axs[1].set(yscale='log', title='Offline action MSE per checkpoint (HOM, matched demo)',
               xlabel='Step (k)', ylabel=MSE_YLABEL)
    axs[1].legend(frameon=False, loc='best')
    fig.suptitle('E1  Training loss & held-out matched action MSE', y=1.02)
    fig.tight_layout()
    register('E1_T1_loss_and_action_mse', fig,
             'Template §4.1 layout. Left: train loss; right: HOM generation MSE (≠ flow loss scale).')

    # ckpt table like template
    rows = []
    for name, ctx in CTX.items():
        tr, hom = ctx['train'], ctx['hom']
        steps = sorted(hom['step'].astype(int)) if not hom.empty else []
        for s in steps:
            loss_win = tr[(tr.step > s - 1000) & (tr.step <= s)]['loss'].mean() if not tr.empty else np.nan
            h = hom[hom.step == s].iloc[0]
            rows.append({
                'run': name, 'ckpt': s,
                'train_loss_1k': float(loss_win) if np.isfinite(loss_win) else np.nan,
                'matched': float(h['matched']),
                'blank_ratio': float(h['blank_ratio']) if np.isfinite(h['blank_ratio']) else np.nan,
                'alt_ratio': float(h['alt_ratio']) if np.isfinite(h.get('alt_ratio', np.nan)) else np.nan,
                'wrong_ratio': float(h['wrong_ratio']) if np.isfinite(h['wrong_ratio']) else np.nan,
            })
    TABLE = pd.DataFrame(rows)
    if not TABLE.empty:
        print('E1 ckpt table (train_loss = mean of prior 1k steps):')
        display(TABLE.round(4))

In [ ]:
# ===== C1 / C2 (stage_b_compare style) =====
def evaluation_series(ax, df, metric, color=None, offset=0, label=''):
    df = df.sort_values('step')
    ax.plot(df['step'] + offset, df[metric], marker='o', ms=5, lw=1.6, color=color,
            label=f'{label}{metric}; {int(df["queries"].iloc[-1]) if "queries" in df and len(df) else "?"} queries')


def ratio_series(ax, df, condition, color=None, offset=0, label='', ci_label=True):
    key, low, high = f'{condition}_ratio', f'{condition}_ci_low', f'{condition}_ci_high'
    if key not in df or df[key].isna().all():
        return
    color = color or COLORS[condition]
    df = df.sort_values('step')
    x = (df['step'] + offset).to_numpy()
    q = int(df['queries'].iloc[-1]) if 'queries' in df and len(df) else 0
    ax.plot(x, df[key], marker='o', ms=5, lw=1.6, color=color,
            label=f'{label}{condition}; {q} queries')
    if low in df and high in df and df[low].notna().any():
        ax.fill_between(x, df[low].to_numpy(dtype=float), df[high].to_numpy(dtype=float),
                        color=color, alpha=0.18,
                        label=('within-checkpoint paired bootstrap 95% CI' if ci_label else None))


if CTX:
    fig, ax = plt.subplots(figsize=(11, 4.8))
    for name, ctx in CTX.items():
        tr = ctx['train']
        if tr.empty:
            continue
        color = RUN_COLORS.get(name, '#333')
        ax.plot(tr['step'] + ctx['start'], tr['loss'], color=color, lw=0.7, alpha=0.25,
                label=f'{name} logged {ctx["freq"]}-step mean')
        ax.plot(tr['step'] + ctx['start'], train_smooth(tr, 'loss', ctx['freq']), color=color, lw=1.8,
                label=f'{name} trailing {SMOOTH_STEPS:,}-step mean')
    finish_axis(ax, 'C1 Training flow-matching loss', 'Flow loss (unitless)', 'Step')
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: step_k(x)))
    fig.tight_layout()
    register('C1_train_loss', fig, 'Rank-0 loss.log; pale logged + solid trailing mean.')

    fig, ax = plt.subplots(figsize=(11, 4.8))
    for name, ctx in CTX.items():
        if ctx['hom'].empty:
            continue
        evaluation_series(ax, ctx['hom'], 'matched', color=RUN_COLORS.get(name, COLORS['matched']),
                          offset=ctx['start'], label=f'{name} ')
    finish_axis(ax, 'C2 Held-out matched action error', MSE_YLABEL, 'Step')
    ax.set_yscale('log')
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: step_k(x)))
    fig.tight_layout()
    register('C2_matched', fig, 'HOM generation MSE on held-out episodes (matched demo).')

    # C3 — same layout as stage_b_compare blank/alt/wrong panel
    conditions = [c for c in ('blank', 'alt', 'wrong')
                  if any((not ctx['hom'].empty and f'{c}_ratio' in ctx['hom']
                          and ctx['hom'][f'{c}_ratio'].notna().any()) for ctx in CTX.values())]
    if conditions:
        fig, axs = plt.subplots(1, len(conditions), figsize=(FIGSIZE_RATIO[0] * len(conditions), FIGSIZE_RATIO[1]),
                               squeeze=False)
        for ax, cond in zip(axs[0], conditions):
            for i, (name, ctx) in enumerate(CTX.items()):
                ratio_series(ax, ctx['hom'], cond, color=RUN_COLORS.get(name, COLORS[cond]),
                             offset=ctx['start'], label=f'{name} ', ci_label=(i == 0))
            ratio_refs(ax, cond)
            finish_axis(ax, f'{cond.capitalize()} demo / matched error ratio', RATIO_YLABEL, 'Step')
            ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: step_k(x)))
        fig.suptitle('Stage A HOM: blank / alt / wrong ratios', y=1.02)
        fig.tight_layout()
        register('C3_g9_ratios', fig,
                 'Shaded: within-checkpoint paired bootstrap 95% CI. Ratio>1 means matched demo helps.')

In [ ]:
if EXPORT and FIGURES:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    meta = {
        'generated_at': datetime.now(timezone.utc).isoformat(),
        'runs': {n: {'path': str(c['run']), 'hom_n': len(c['hom'])} for n, c in CTX.items()},
        'notes': NOTES,
    }
    (EXPORT_DIR / 'meta.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')
    if 'TABLE' in dir() and isinstance(TABLE, pd.DataFrame) and not TABLE.empty:
        TABLE.to_csv(EXPORT_DIR / 'e1_ckpt_table.csv', index=False)
    for name, fig in FIGURES.items():
        for ext in FORMATS:
            fig.savefig(EXPORT_DIR / f'{name}.{ext}', dpi=DPI if ext == 'png' else None, bbox_inches='tight')
    print(f'exported {len(FIGURES)} figures → {EXPORT_DIR}')
elif not FIGURES:
    print('无图可导出：需要 loss.log 和/或 heldout_monitor.jsonl')